# 0회차 — Introduction: 수학 한 줄이 코드 세 줄·그림 한 장으로

> Part 1 · 0회차 (Introduction)
> 핵심 시연: **SVD로 이미지 압축하기**

## 이 노트북의 약속

이 노트북 한 개로 다음 사실을 체득한다.

1. **이미지는 행렬이다** — $A \in \mathbb{R}^{m \times n}$
2. **임의의 행렬은 세 행렬의 곱으로 분해된다** — $A = U \Sigma V^\top$ (SVD)
3. **분해된 부분 일부만 남기면 원본에 가까운 압축이 된다** — $A_k = U_{:,1:k}\,\Sigma_{1:k,1:k}\,V_{:,1:k}^\top$
4. **이 한 줄의 수학이 NumPy 한 줄로 실행된다** — `U, s, Vt = np.linalg.svd(A)`
5. **결과는 시각적으로 즉시 확인된다** — k가 작으면 흐릿, 크면 선명

오늘은 정의·증명을 다 알 필요 없다. **"수학 정의 한 줄이 코드와 그림으로 직결된다"는 감각만** 가져가면 충분하다. 정식 정의는 1회차부터, SVD의 본격 다룸은 Part 2 5회차에서.

---

## 이 한 시연 안에 들어 있는 25회차의 지점

| 식의 부분 | 회차 |
|---|---|
| $A \in \mathbb{R}^{m\times n}$ — 이미지를 행렬로 본다 | 1·2회차 |
| $U, V$ 두 직교 행렬 — 회전 | 10·11회차 |
| $\Sigma$ 대각 행렬 — 축별 신축 | Part 2 1·5회차 |
| $A_k$ 저계수 근사 (Eckart-Young) | Part 2 5-6회차 |
| 압축률 vs 화질의 균형 | Part 2 5·6회차 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
print('NumPy:', np.__version__)

## 1. 이미지를 행렬로 — 단 한 줄

MNIST 손글씨 한 장을 가져온다 (8×8 mini-MNIST, 의존성 없음). 
더 인상적인 화질이 필요하면 `scipy.misc.face()` 또는 본인 이미지로 교체.

In [ ]:
# 옵션 A — sklearn mini-MNIST (8×8, 항상 사용 가능)
from sklearn.datasets import load_digits
digits = load_digits()
A_small = digits.images[0]  # 8×8

# 옵션 B — scipy의 face (768×1024 컬러). 더 인상적인 화질 시연용
try:
    from scipy.datasets import face
    A_color = face(gray=False).astype(float)
    A_big = face(gray=True).astype(float)
    BIG_AVAILABLE = True
except Exception:
    BIG_AVAILABLE = False
    print('scipy.datasets.face() 사용 불가 — mini-MNIST로 진행')

if BIG_AVAILABLE:
    A = A_big   # 큰 이미지로 시연
    print(f'이미지 크기: {A.shape[0]} × {A.shape[1]} = {A.size:,} 픽셀')
else:
    A = A_small.astype(float)
    print(f'이미지 크기: {A.shape[0]} × {A.shape[1]} = {A.size:,} 픽셀 (mini-MNIST)')

plt.figure(figsize=(5, 4))
plt.imshow(A, cmap='gray')
plt.title(f'원본 이미지 — 행렬 A ∈ ℝ^{A.shape[0]}×{A.shape[1]}')
plt.axis('off')
plt.show()

**관찰**: 

이미지를 행렬로 본다는 것은 곧 **각 픽셀이 행렬의 원소 한 개**라는 의미. 
이 행렬은 $m \times n$ 차원이고, 픽셀 수는 $mn$. 

이 표현이 정당화되는 이유는 **벡터공간의 정의**(MML §2.4)에서 따라 나온다 — 5회차에서 본다. 
지금은 "이미지 = 숫자 행렬"만 받아들인다.

## 2. SVD — 수학 정의 한 줄

### 정리 (SVD, Part 2 5회차에서 증명)

임의의 행렬 $A \in \mathbb{R}^{m\times n}$은 다음과 같이 분해된다.

$$A = U\Sigma V^\top$$

- $U \in \mathbb{R}^{m\times m}$: 직교 행렬 (회전)
- $\Sigma \in \mathbb{R}^{m\times n}$: 대각 행렬, 대각 원소 $\sigma_1 \ge \sigma_2 \ge \cdots \ge 0$ (신축 — 특이값)
- $V \in \mathbb{R}^{n\times n}$: 직교 행렬 (회전)

### 코드 한 줄

In [ ]:
U, s, Vt = np.linalg.svd(A, full_matrices=False)

print(f'U shape : {U.shape}')
print(f's shape : {s.shape}   (특이값 벡터, len = min(m, n))')
print(f'Vt shape: {Vt.shape}  (V^T)')
print(f'특이값 처음 10개: {s[:10].round(2)}')
print(f'특이값 마지막 10개: {s[-10:].round(2)}')

**관찰**:

- 특이값들이 **큰 순서로 정렬**되어 있다 (NumPy 규약)
- 큰 특이값일수록 "행렬의 본질적 구조"를 많이 담는다
- 작은 특이값들은 노이즈·디테일에 해당

이 사실을 시각적으로 확인하자.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(s, marker='o', markersize=3)
axes[0].set_title('특이값 σ_i (선형 스케일)')
axes[0].set_xlabel('i')
axes[0].set_ylabel('σ_i')
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(s, marker='o', markersize=3)
axes[1].set_title('특이값 σ_i (로그 스케일)')
axes[1].set_xlabel('i')
axes[1].set_ylabel('σ_i (log)')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 누적 에너지: 상위 k개 σ²의 합 / 전체 합
cum_energy = np.cumsum(s**2) / (s**2).sum()
print(f'상위 10개 특이값이 차지하는 에너지: {cum_energy[9]*100:.2f}%')
print(f'상위 50개 특이값이 차지하는 에너지: {cum_energy[min(49, len(s)-1)]*100:.2f}%')
print(f'상위 100개 특이값이 차지하는 에너지: {cum_energy[min(99, len(s)-1)]*100:.2f}%')

**관찰**:

전체 특이값 중 **상위 몇 십 개**가 거의 모든 에너지를 차지한다. 
→ 나머지는 버려도 큰 손실 없다는 직관. 
이게 **저계수 근사**(low-rank approximation)의 핵심 아이디어.

## 3. 압축 — rank-k 근사

### 정의 (rank-k 근사)

$A = U\Sigma V^\top$에서 상위 $k$개 특이값만 남기고 나머지를 0으로:

$$A_k = \sum_{i=1}^{k} \sigma_i\, \mathbf{u}_i\, \mathbf{v}_i^\top = U_{:,1:k}\,\Sigma_{1:k,1:k}\,V_{:,1:k}^\top$$

### 정리 (Eckart-Young, Part 2 5회차)

$A_k$는 **rank가 $k$인 모든 행렬 중 $A$에 가장 가까운 행렬**이다 (Frobenius 노름 기준).

$$\|A - A_k\|_F = \sqrt{\sigma_{k+1}^2 + \sigma_{k+2}^2 + \cdots}$$

즉 "버린 작은 특이값의 제곱합"이 오차다.

### 코드

In [ ]:
def rank_k_approx(U, s, Vt, k):
    """상위 k 특이값으로 rank-k 근사 재구성"""
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]


# 적당한 k 후보 — 이미지 크기에 따라 자동 선택
min_dim = min(A.shape)
if min_dim >= 200:
    ks = [1, 5, 20, 50, 100, min_dim]
elif min_dim >= 50:
    ks = [1, 3, 5, 10, 20, min_dim]
else:
    ks = [1, 2, 3, 4, 6, min_dim]

fig, axes = plt.subplots(1, len(ks), figsize=(3 * len(ks), 3.2))
for ax, k in zip(axes, ks):
    A_k = rank_k_approx(U, s, Vt, k)
    ax.imshow(A_k, cmap='gray', vmin=A.min(), vmax=A.max())
    err = np.linalg.norm(A - A_k, 'fro') / np.linalg.norm(A, 'fro')
    ax.set_title(f'k={k}\nrel.err={err:.3f}')
    ax.axis('off')
plt.suptitle(f'rank-k 근사 비교 (전체 특이값 {min_dim}개)')
plt.tight_layout()
plt.show()

**관찰**:

- $k=1$: 가장 강한 한 방향만 — 거의 흐릿한 그림자
- $k$가 커질수록 점점 원본에 가까워짐
- 어느 시점부터는 $k$를 더 늘려도 시각적 차이가 거의 없음

이 "시각적 차이가 사라지는 시점"이 곧 **유효 차원(intrinsic dimension)**의 직관. 
8회차의 일차독립·차원, Part 2 5회차의 PCA가 이 직관의 정식화.

## 4. 압축률 vs 화질 — 저장 공간 분석

$m \times n$ 행렬 $A$를 저장하려면 $mn$개의 수가 필요하다. 
$A_k$는 $U_{:,1:k}$ ($mk$개), $\Sigma_{1:k}$ ($k$개), $V_{:,1:k}$ ($nk$개)만 저장하면 되므로 **총 $k(m + n + 1)$개**.

압축비:

$$\rho(k) = \frac{k(m + n + 1)}{mn}$$

이 값이 1보다 작은 $k$ 범위가 "실제 압축이 이득인" 구간이다.

In [ ]:
m, n = A.shape
total_pixels = m * n
k_range = np.arange(1, min_dim + 1)

compression_ratio = k_range * (m + n + 1) / total_pixels
rel_errors = []
for k in k_range:
    A_k = rank_k_approx(U, s, Vt, k)
    rel_errors.append(np.linalg.norm(A - A_k, 'fro') / np.linalg.norm(A, 'fro'))
rel_errors = np.array(rel_errors)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(k_range, compression_ratio, 'b-', label='저장 비율 ρ(k)')
ax1.axhline(1, color='b', linestyle=':', alpha=0.5, label='원본 동일')
ax1.set_xlabel('k (남긴 특이값 개수)')
ax1.set_ylabel('저장 비율 (압축률)', color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(k_range, rel_errors, 'r-', label='상대 오차')
ax2.set_ylabel('상대 오차 |A−A_k|_F / |A|_F', color='r')
ax2.tick_params(axis='y', labelcolor='r')

plt.title(f'압축률 ↔ 화질 trade-off  (m={m}, n={n})')
ax1.legend(loc='center right')
plt.tight_layout()
plt.show()

# "이득 구간" 찾기 — 압축률 < 0.5이고 오차 < 5%인 k
good_k = np.where((compression_ratio < 0.5) & (rel_errors < 0.05))[0]
if len(good_k):
    print(f'압축률 50% 이내 + 상대오차 5% 이내인 k 범위: {good_k[0]+1} ~ {good_k[-1]+1}')

## 5. 수학 → 코드 → 그림 — 한 사이클 정리

방금 본 과정을 한 줄로 압축하면:

```
수학:  A = UΣV^T            ─ 한 줄 정리
코드:  U, s, Vt = np.linalg.svd(A)
       A_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
그림:  k에 따라 흐릿 → 선명
```

이 흐름이 **이 강의 25회차 전체의 표준 사이클**이다. 매 회차 같은 사이클로 
(정의 → 정리 → NumPy 한 줄 → 시각화) 를 반복한다.

### 오늘 정의·정리 없이 본 것의 정식화는 어디서?

| 오늘 본 것 | 정식 다룸 |
|---|---|
| 이미지가 행렬 | 1·2회차 |
| 직교 행렬 $U, V$ | 10·11회차 |
| 대각 신축 $\Sigma$ | Part 2 1·5회차 |
| rank-k 근사·Eckart-Young | Part 2 5-6회차 |
| Frobenius 노름 | 1·11회차 |

지금은 "수학과 코드가 직결된다"는 감각만 가져가면 충분.


## 6. 보너스 — 컬러 이미지로 확장

컬러 이미지(`H × W × 3`)에 SVD를 적용하려면 R/G/B 채널 각각에 SVD를 적용한다. 
(또는 텐서 분해 — Part 2 6회차에서 잠깐 언급)

In [ ]:
if BIG_AVAILABLE:
    def compress_color(img, k):
        """R, G, B 채널별로 rank-k SVD 압축"""
        out = np.zeros_like(img)
        for c in range(3):
            U_c, s_c, Vt_c = np.linalg.svd(img[..., c], full_matrices=False)
            out[..., c] = (U_c[:, :k] @ np.diag(s_c[:k]) @ Vt_c[:k, :])
        return np.clip(out, 0, 255)

    ks_c = [5, 20, 50, 100]
    fig, axes = plt.subplots(1, len(ks_c) + 1, figsize=(3 * (len(ks_c) + 1), 3))
    axes[0].imshow(A_color.astype(np.uint8))
    axes[0].set_title('원본')
    axes[0].axis('off')
    for ax, k in zip(axes[1:], ks_c):
        ax.imshow(compress_color(A_color, k).astype(np.uint8))
        ax.set_title(f'k={k}')
        ax.axis('off')
    plt.suptitle('컬러 이미지 SVD 압축 (채널별)')
    plt.tight_layout()
    plt.show()
else:
    print('컬러 이미지 시연은 scipy.datasets.face() 가용 시 활성화됩니다.')

## 7. 자가 점검 — 다음 회차 전에

이 노트북을 닫기 전에 답해 보자. (정답을 본인 노트에 한 문장씩)

1. 이미지를 행렬로 보는 것이 정당화되는 이유는 무엇인가? 
   (힌트: 픽셀값의 덧셈·스칼라곱이 자연스럽다 → 벡터공간 공리)

2. SVD에서 $U$와 $V$가 "직교 행렬"이라는 말의 직관은 무엇인가? 
   (힌트: "길이를 보존하는 변환" = "회전·반전")

3. 특이값 $\sigma_i$가 큰 순서로 정렬되어 있다는 것은 무엇을 의미하나? 
   (힌트: "가장 중요한 방향부터")

4. $k$를 어떻게 정하나? 
   (힌트: 누적 에너지 95%·99% 기준 또는 시각적 만족도)

5. 이 압축은 손실 압축인가 무손실 압축인가? 
   (힌트: $k < \mathrm{rank}(A)$이면 정보 손실, $k = \mathrm{rank}(A)$이면 완전 복원)

---

## 다음 회차 (1회차) 예고

이번에 "이미지 = 행렬"이라고 받아들였는데, 그 정당화는 **벡터 공리**에서 시작한다. 
1회차에서 벡터의 정의·노름·내적을 수학적으로 쌓는다. 

사전 reading:
- MML §2.1, §3.1, §3.2 
- Strang §1.1, §1.2 
- 3Blue1Brown EoLA Ch.1, Ch.2